In [33]:
import json
import pickle
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

In [34]:
data_path = Path('../data/Modena')

images_paths = sorted(data_path.rglob('*.jpg'))
irr_data_paths = sorted(data_path.rglob('*.json'))

image_dir = data_path / 'images'
image_dir.mkdir(exist_ok=True)

In [35]:
# for img_path in tqdm(images_paths):
#     filename = img_path.name

#     sec = int(filename[18:20])
#     if sec < 30:
#         filename = filename[:18] + '00' + filename[20:]
#     else:
#         print(f'Image path {img_path} has invalid second value: {sec}')
#         continue

#     dst = image_dir / filename

#     shutil.move(img_path, dst)

In [36]:
irr_data_list = []

for irr_data_path in tqdm(irr_data_paths):
    filename = irr_data_path.name
    sec = int(filename[18:20])
    if sec < 30:
        filename = filename[:18] + '00' + filename[20:]
    else:
        print(f'Irradiance readings path {irr_data_path} has invalid second value: {sec}')
        continue

    with open(irr_data_path, 'r') as f:
        irr_data = json.load(f)

    if irr_data['irradiance'] < 0:
        irr_data['irradiance'] = 0

    irr_data['datetime'] = filename.split('_hukseflux')[0]
    file_extension = '.jpg' if data_path.name == 'Modena' else '_600.jpg'
    irr_data['image_name'] = str(image_dir / Path(filename.replace('hukseflux_sr05-d1a3', 'rpi_hq_imx477r_fisheye_rgb_camera').replace('.json', file_extension)))
    if Path(irr_data['image_name']).exists():
        irr_data_list.append(irr_data)

  0%|          | 0/9510 [00:00<?, ?it/s]

  9%|▉         | 871/9510 [00:00<00:01, 4344.52it/s]

Irradiance readings path ../data/Modena/hukseflux_sr05-d1a3/2024/06/06/2024_06_06__17_19_59_hukseflux_sr05-d1a3.json has invalid second value: 59
Irradiance readings path ../data/Modena/hukseflux_sr05-d1a3/2024/06/06/2024_06_06__17_37_53_hukseflux_sr05-d1a3.json has invalid second value: 53
Irradiance readings path ../data/Modena/hukseflux_sr05-d1a3/2024/06/06/2024_06_06__18_51_34_hukseflux_sr05-d1a3.json has invalid second value: 34


100%|██████████| 9510/9510 [00:01<00:00, 4893.77it/s]


In [37]:
df = pd.DataFrame.from_dict(irr_data_list)
df.to_csv(data_path / f'{data_path.name.lower()}_irradiance.csv', index=False)
df = pd.read_csv(
    data_path / f'{data_path.name.lower()}_irradiance.csv',
    parse_dates=['datetime'],
    date_format='%Y_%m_%d__%H_%M_%S',
    index_col='datetime',
)

print(f'Readings: {len(df)}')
print(f'Min irradiance: {df["irradiance"].min()}')
print(f'Max irradiance: {df["irradiance"].max()}')
print(f'Mean irradiance: {df["irradiance"].mean()}')
print(f'Irradiance std: {df["irradiance"].std()}')

Readings: 9474
Min irradiance: 0.0
Max irradiance: 1257.31
Mean irradiance: 228.27537365421153
Irradiance std: 298.2517989672098


In [38]:
time_shift = 5
history_size = 4
time_window = 15

output_path = data_path / Path(f'{data_path.name.lower()}_periods_{history_size}_{time_shift}_{time_window}.pkl')

periods = []

for t, _ in tqdm(df.iterrows(), total=len(df)):
    t = pd.Timestamp(t)
    t_history = [t - pd.Timedelta(minutes=time_shift * i) for i in range(history_size - 1, 0, -1)]
    t_target = t + pd.Timedelta(minutes=time_window)

    if all(map(lambda x: x in df.index, [*t_history, t_target])):
        if df.loc[t_target]['irradiance'] >= 0.0:
            periods.append({
                'history': [
                    {
                        'image_name': df.loc[_t]['image_name'],
                        'irradiance': df.loc[_t]['irradiance']
                    } for _t in sorted([*t_history, t])
                ],
                'target_irradiance': df.loc[t_target]['irradiance'],
            })

print(f'Number of periods: {len(periods)}')
with output_path.open('wb') as f:
    pickle.dump(periods, f)

100%|██████████| 9474/9474 [00:31<00:00, 297.63it/s]


Number of periods: 9420


In [39]:
periods[0]

{'history': [{'image_name': '../data/Modena/images/2024_06_06__18_52_00_rpi_hq_imx477r_fisheye_rgb_camera.jpg',
   'irradiance': 114.08},
  {'image_name': '../data/Modena/images/2024_06_06__18_57_00_rpi_hq_imx477r_fisheye_rgb_camera.jpg',
   'irradiance': 71.55},
  {'image_name': '../data/Modena/images/2024_06_06__19_02_00_rpi_hq_imx477r_fisheye_rgb_camera.jpg',
   'irradiance': 76.28},
  {'image_name': '../data/Modena/images/2024_06_06__19_07_00_rpi_hq_imx477r_fisheye_rgb_camera.jpg',
   'irradiance': 70.49}],
 'target_irradiance': 55.39}